# HBT / $g^{(2)}$ Analysis — Guided Tutorial

A complete, runnable walkthrough of the second-order correlation (HBT) pipeline for the High-Harmonic
Generation (HHG) experiments. By the end you will know **what is stored in the `.pkl` files**, **how every
metric is computed and from which key**, and **how the figures are produced**.

Two engines power everything:
* `src/hbt_core.py` — `HBTMeasurement`: loads a run, exposes the data, computes $g^{(2)}$ and $R$.
* `src/hbt_visu.py` — `GridVisualizer`: builds the coherence / $g^{(2)}$ / $R$ single plots and grids.

**The single most important idea** is the distinction between *physical* and *virtual* data — see §3.

## 0. Setup

In [ ]:
import numpy as np
from pathlib import Path

# Importing hbt_visu also configures matplotlib (serif + LaTeX) for publication-style figures.
from src.hbt_core import HBTMeasurement
from src.hbt_visu import GridVisualizer

## 1. The datasets

Two kinds of files appear in this project:

* **`data/Jun15/test.pkl`** — the *new* acquisition format. Besides the scalars it stores the genuine
  per-detector histograms `correlations_physical`, i.e. real **physical coherence**.
* **`data/Jun12/...`** — four *older* runs (filters 700/10 vs 700/40 nm, polariser P1 vs none). These store
  only the merged-harmonic (`correlations_virtual`) histograms.

The code adapts to both automatically.

In [ ]:
new_path = Path("data/Jun15/test.pkl")

jun12 = Path("data/Jun12")
compare_paths = {
    "700-10 (No Pol)": jun12 / "700-10 no P1 2026-06-12_17-36-40_g2_heralded_virtual" / "17-36-40_2026-06-12_40.7mW_num0_chunk0.pkl",
    "700-10 (Pol 1)":  jun12 / "700-10 P1 2026-06-12_17-10-59_g2_heralded_virtual"    / "17-10-59_2026-06-12_40.7mW_num0_chunk0.pkl",
    "700-40 (No Pol)": jun12 / "700-40 no P1 2026-06-12_17-30-07_g2_heralded_virtual" / "17-30-07_2026-06-12_40.7mW_num0_chunk0.pkl",
    "700-40 (Pol 1)":  jun12 / "700-40 P1 2026-06-12_17-26-52_g2_heralded_virtual"    / "17-26-52_2026-06-12_40.7mW_num0_chunk0.pkl",
}

run = HBTMeasurement(new_path)
print("Loaded:", run.short_tag(), "| physical histograms:", run.has_physical_histograms)

## 2. Look inside the raw `.pkl`

`describe()` walks the whole pickle and prints every key with its type, array length (+ preview) or value.

In [ ]:
run.describe(max_depth=4)

## 3. Physical vs virtual — the data model (the big idea)

Each harmonic $H_n$ hits a **beamsplitter** and is detected on two physical detectors, *Transmitted* `HnT`
and *Reflected* `HnR`:

| Detector | 1 | 2 | 3 | 4 | 5 | 6 |
|----------|----|----|----|----|----|----|
| Mode     | H3T | H3R | H4T | H4R | H5T | H5R |

A **virtual** channel $H_n$ is the *merged* (T + R) click stream of one harmonic.

What is stored:

| Quantity | Physical (per detector) | Virtual (merged harmonic) |
|---|---|---|
| Singles | `counts_physical`, `countrates_physical` | `counts_virtual`, `countrates_virtual` |
| Two-fold coincidences (scalar) | `coincidences_twofold_physical` (15 pairs) | `coincidences_twofold_virtual` (3 pairs) |
| Full $g^{(2)}(\tau)$ histogram | `correlations_physical` **(new files only)** | `correlations_virtual` (always) |
| Three-fold heralded | — | `heralded_threefold` |

**How the code routes the data (and the "fallback").**
The physical $g^{(2)}(0)$ from the **scalars** (`compute_g2_direct`) is always available and artifact-free.
For the **histogram** methods (coherence, `delay`) the data is chosen by a `source` argument:

* `source='auto'` (default) → use `correlations_physical` if present, otherwise `correlations_virtual`,
* `source='physical'` / `'virtual'` → force one, with **no silent substitution**.

This replaces an earlier bug where a missing physical histogram was silently swapped for a virtual one — which,
for **auto**-correlations, injected a non-physical self-coincidence spike at $\tau=0$ ($g^{(2)}\!\approx\!3$
instead of the true value) and collapsed all T/R combinations onto a single curve.

In [ ]:
print("has_physical_histograms :", run.has_physical_histograms)
print("has_virtual_histograms  :", run.has_virtual_histograms)
print("resolve_source('auto')  :", run.resolve_source('auto'), "<- what the trace methods will read")

## 4. Channels, harmonics & metadata

The driving laser is at 2100 nm, so $H_3, H_4, H_5 \approx$ 700, 525, 420 nm. `channel_map`/`get_ch` translate
between detector numbers and names; helper attributes feed the plot titles.

In [ ]:
print("channel_map :", run.channel_map)
print("get_ch('H3T'):", run.get_ch("H3T"), "| get_ch('H4R'):", run.get_ch("H4R"))
print()
print("material / wavelength / power :", run.material, "/", run.wavelength_nm, "nm /", run.power_mw, "mW")
print("rep rate / period            :", run.rep_rate_hz/1e6, "MHz /", round(run.rep_period_ns,3), "ns")
print("bin / coincidence window     :", run.binwidth_ps, "ps /", run.coincidence_window_ns, "ns")
print()
print("essentials :", run.acquisition_essentials())
print("details    :", run.acquisition_details())

## 5. Building blocks: singles, coincidences, pulses

* **Singles** $N_i$ = `counts_physical[str(c)]` (`_get_counts`).
* **Two-fold coincidences** $N_{12}$ = `coincidences_twofold_physical['(c1,c2)']` (`_get_twofold_coincidence`).
* **Number of pulses** $N_\mathrm{pulse} = T_\mathrm{acq}\,f_\mathrm{rep}$.

In [ ]:
c1, c2 = run.get_ch("H3T"), run.get_ch("H3R")
N1, N2 = run._get_counts(c1), run._get_counts(c2)
N12 = run._get_twofold_coincidence(c1, c2)
N_pulse = (run.duration * 1e-12) * run.rep_rate_hz
print(f"N1 = {N1:,.0f}   N2 = {N2:,.0f}   N12 = {N12:,.0f}   N_pulse = {N_pulse:,.0f}")

## 6. Physical $g^{(2)}(0)$ — and a cross-check

The **direct** (scalar) physical estimate:
$$ g^{(2)}(0) = \frac{N_\mathrm{pulse}\,N_{12}}{N_1 N_2} $$

Because this file also has physical *histograms*, we can validate it against the **delay** (peak-area-ratio)
method on the same physical data. They should agree for the auto-correlations.

In [ ]:
def gd(a, b):  # physical direct (scalar)
    return run.compute_g2_direct(run.get_ch(a), run.get_ch(b))
def gl(a, b):  # physical delay (histogram)
    return run.compute_g2_delay(run.get_ch(a), run.get_ch(b), 5.0, source='physical')

print(f"{'pair':16s}{'direct':>10s}{'delay':>10s}")
for name,(a,b) in {"H3 auto":("H3T","H3R"),"H4 auto":("H4T","H4R"),"H5 auto":("H5T","H5R"),
                   "H3-H4":("H3T","H4T"),"H3-H5":("H3T","H5T"),"H4-H5":("H4T","H5T")}.items():
    print(f"{name:16s}{gd(a,b):10.4f}{gl(a,b):10.4f}")

## 7. Histogram methods & the self-correlation artifact

`get_correlation_trace(c1, c2, source=...)` returns `(tau_ns, counts)`. With physical histograms the auto pair
`(HnT, HnR)` is a genuine T$\times$R cross-correlation — a clean peak, no $\tau=0$ spike. (On old virtual-only
files, the virtual auto carries a self-coincidence spike that the code suppresses automatically.)

In [ ]:
import matplotlib.pyplot as plt
x, y = run.get_correlation_trace(run.get_ch("H3T"), run.get_ch("H3R"))   # physical auto (source='auto')
i0 = int(np.argmin(np.abs(x)))
print("physical H3 auto, around tau=0:", [int(y[j]) for j in range(i0-2, i0+3)], " (smooth peak, no spike)")

fig, ax = plt.subplots(figsize=(8,4), dpi=120)
m = (x > -120) & (x < 120)
ax.plot(x[m], y[m]*1e-3, lw=1)
ax.set_xlabel(r"$\Delta t$ (ns)"); ax.set_ylabel(r"Counts $\times 10^3$")
ax.set_title("Raw physical H3 auto-correlation (H3T x H3R)")
ax.grid(alpha=0.3); plt.show()

## 8. Cauchy–Schwarz parameter $R$

$$ R = \frac{[g^{(2)}_{\mathrm{cross}}]^2}{g^{(2)}_{\mathrm{auto},1}\,g^{(2)}_{\mathrm{auto},2}}, \qquad R>1 \;\Rightarrow\; \text{non-classical}. $$

In [ ]:
gc = run.compute_g2_direct(run.get_ch("H3T"), run.get_ch("H4T"))
ga = run.compute_g2_direct(run.get_ch("H3T"), run.get_ch("H3R"))
gb = run.compute_g2_direct(run.get_ch("H4T"), run.get_ch("H4R"))
print(f"g2_cross(H3,H4)={gc:.4f}  g2_auto(H3)={ga:.4f}  g2_auto(H4)={gb:.4f}  ->  R_34={run.compute_R_parameter(gc,ga,gb):.4f}")

## 9. Visualisation gallery — single plots

Titles and legends are filled from the metadata. With physical histograms present, the coherence plot is
labelled *(physical)* and the `delay` method is labelled *Physical (delay)*.

In [ ]:
visu = GridVisualizer(run)
H3T, H3R, H4T, H4R = (run.get_ch(n) for n in ("H3T","H3R","H4T","H4R"))

# Coherence (physical) for an auto and a cross pair.
visu.plot_coherence(H3T, H3R, xlim=150, integration_window_ns=10)
visu.plot_coherence(H3T, H4T, xlim=150, integration_window_ns=10)

In [ ]:
# g2 integration sweep (all methods) and a single R sweep.
visu.plot_g2(H3T, H4T, methods=['direct', 'delay', 'heralded'], tau_min=0.3, tau_max=40, step=1.0)
visu.plot_R(cross_pair=(H3T, H4T), auto_pair_1=(H3T, H3R), auto_pair_2=(H4T, H4R),
            methods=['direct', 'delay'], tau_min=0.3, tau_max=40, step=1.0)

## 10. Visualisation gallery — full grids

Call the same methods with no channel pair to get the master matrices: coherence (5x3), $g^{(2)}$ (5x3) and
Cauchy–Schwarz $R$ (4x3).

In [ ]:
visu.plot_coherence(time_window_ns=5, xlim=150, integration_window_ns=10)

In [ ]:
visu.plot_g2(methods=['direct', 'delay'], tau_min=0.3, tau_max=40, step=1.5)

In [ ]:
visu.plot_R(methods=['direct', 'delay'], tau_min=0.3, tau_max=40, step=1.5)

## 11. Comparing runs — filters & polarisation (older files)

Overlay the four June-12 runs. With no explicit labels, each curve is auto-tagged
`filter | polarisation | power` from its own metadata. These files have no physical histograms, so the
`delay`/coherence data is virtual (and labelled as such), while `direct` stays physical.

In [ ]:
runs = [HBTMeasurement(p) for p in compare_paths.values()]
comparator = GridVisualizer(runs, comparison_variable="Filter \\& Polarisation")
print("histogram source for this comparison:", comparator.histogram_source)

comparator.plot_g2(methods=['direct'], tau_min=0.3, tau_max=25, step=1.0)
comparator.plot_R(methods=['direct'], tau_min=0.3, tau_max=20, step=1.0)

## Cheat-sheet

| Goal | Call | Reads from the pkl |
|---|---|---|
| Inspect the file | `run.describe()` | everything |
| Which histograms exist | `run.has_physical_histograms`, `run.resolve_source('auto')` | — |
| Physical $g^{(2)}(0)$ *(default)* | `run.compute_g2_direct(c1, c2)` | `coincidences_twofold_physical`, `counts_physical`, `duration`, `rep_rate` |
| Histogram $g^{(2)}$ (area ratio) | `run.compute_g2_delay(c1, c2, tau, source='auto')` | `correlations_physical` or `correlations_virtual` |
| Heralded $g^{(2)}$ | `run.compute_g2_heralded(c1, c2, tau)` | `correlations_virtual`, `counts_virtual` |
| Cauchy–Schwarz $R$ | `run.compute_R_parameter(gc, ga, gb)` | (from the $g^{(2)}$ above) |
| Histogram trace | `run.get_correlation_trace(c1, c2, source='auto')` | physical if present, else virtual |
| Figures | `GridVisualizer(run).plot_coherence / plot_g2 / plot_R` | all of the above |

**Golden rule:** `direct` is always physical; the histogram methods use physical data when it exists and
are clearly labelled *physical* vs *virtual* so a plot is never mislabelled.